# OmniParser 실습: IKEA 조립 설명서 적용

GUI 화면용 OmniParser 공개 코드를 조립 설명서 그림에 적용하는 실습

- 환경: Google Colab, T4 GPU (런타임 -> 런타임 유형 변경 -> T4 GPU)
- 레포: https://github.com/microsoft/OmniParser
- 가중치: https://huggingface.co/microsoft/OmniParser-v2.0
- 위에서부터 순서대로 실행. 세션이 끊기면 1번부터 다시

## 1. 레포 받기, 설치, PaddleOCR 제거

- requirements.txt에는 윈도우 전용, 에이전트용 패키지가 섞여 있어 파싱에 필요한 것만 설치
- PaddleOCR 옛 버전은 Python 3.13에 설치되지 않음 -> utils.py에서 PaddleOCR 부분을 떼고 EasyOCR만 사용

In [ ]:
!git clone https://github.com/microsoft/OmniParser
%cd /content/OmniParser
!pip -q install easyocr supervision==0.18.0 ultralytics einops timm dill openai pymupdf "transformers==4.49.0"

import re
p = 'util/utils.py'
s = open(p).read()
s = s.replace("from paddleocr import PaddleOCR\n", "")
s = re.sub(r"paddle_ocr = PaddleOCR\(.*?rec_batch_num=1024\)\n", "paddle_ocr = None\n", s, flags=re.S)
open(p, 'w').write(s)
print('PaddleOCR 남아있음:', 'PaddleOCR' in s)

## 2. 가중치 받기

- huggingface-cli는 없어졌으므로 hf 사용
- 탐지 모델: YOLOv9-E (icon_detect_v3), 설명 모델: Florence-2 (icon_caption)

In [ ]:
%%bash
cd /content/OmniParser
hf download microsoft/OmniParser-v2.0 icon_detect_v3/model.pt --revision refs/pr/37 --local-dir weights
hf download microsoft/OmniParser-v2.0 icon_caption/config.json icon_caption/generation_config.json icon_caption/model.safetensors --local-dir weights
mv weights/icon_caption weights/icon_caption_florence
ls weights

## 3. 모델 올리기

In [ ]:
%cd /content/OmniParser
import io, base64, os
import pandas as pd
from PIL import Image
from util.utils import get_som_labeled_img, check_ocr_box, get_caption_model_processor, get_yolo_model

som_model = get_yolo_model(device='cuda')
caption_model_processor = get_caption_model_processor(
    model_name="florence2", model_name_or_path="weights/icon_caption_florence", device='cuda')
os.makedirs('out', exist_ok=True)
print('모델 로드 완료')

## 4. 함수

- run: 이미지 한 장을 파싱해서 번호 그림과 요소 목록을 out 폴더에 저장
- show_crops: 박스를 잘라낸 그림과, 설명 모델이 실제로 받는 64x64 그림을 나란히 출력
- 글자가 하나도 없는 이미지에서 원본 코드가 에러를 내므로 빈 글자 박스를 하나 넣어 우회
- BOX_TRESHOLD는 원본 코드의 철자 그대로

In [ ]:
def run(image_path, tag='base', box_threshold=0.05, scale_img=False, imgsz=None):
    image = Image.open(image_path).convert('RGB')
    r = max(image.size) / 3200
    cfg = {'text_scale': 0.8*r, 'text_thickness': max(int(2*r), 1),
           'text_padding': max(int(3*r), 1), 'thickness': max(int(3*r), 1)}
    try:
        (text, ocr_bbox), _ = check_ocr_box(
            image, display_img=False, output_bb_format='xyxy',
            easyocr_args={'paragraph': False, 'text_threshold': 0.9}, use_paddleocr=False)
    except Exception as e:
        print('OCR 실패:', e)
        text, ocr_bbox = [], []
    if len(ocr_bbox) == 0:
        text, ocr_bbox = [''], [[0, 0, 2, 2]]
    img_b64, _, elems = get_som_labeled_img(
        image, som_model, BOX_TRESHOLD=box_threshold, output_coord_in_ratio=True,
        ocr_bbox=ocr_bbox, draw_bbox_config=cfg,
        caption_model_processor=caption_model_processor, ocr_text=text,
        use_local_semantics=True, iou_threshold=0.7,
        scale_img=scale_img, imgsz=imgsz, batch_size=128)
    labeled = Image.open(io.BytesIO(base64.b64decode(img_b64)))
    df = pd.DataFrame(elems)
    df.insert(0, 'ID', range(len(df)))
    name = os.path.splitext(os.path.basename(image_path))[0]
    labeled.save(f'out/{name}_{tag}.png')
    df.to_csv(f'out/{name}_{tag}.csv', index=False)
    print(name, tag, '| 요소 수:', len(df), '| icon:', (df['type'] == 'icon').sum())
    return labeled, df


def show_crops(image_path, df, ids):
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    for i in ids:
        x1, y1, x2, y2 = df.loc[i, 'bbox']
        crop = img.crop((int(x1*w), int(y1*h), int(x2*w), int(y2*h)))
        print(i, '|', df.loc[i, 'content'])
        display(crop)
        display(crop.resize((64, 64)))

print('함수 준비 완료')

## 5. 대조군: GUI 화면

레포 예시 스크린샷으로 정상 동작 확인. 여기서 되는데 설명서에서 안 되면 데이터 차이로 판단

In [ ]:
labeled, df = run('imgs/word.png')
display(labeled)
df

## 6. 설명서 올리고 그림으로 변환

- 파일 선택 창에서 IKEA 조립 설명서 PDF 선택
- 변환 후 ikea 폴더에서 2장 고르기 (A 부품 목록, B 조립 단계)
- 결과를 보기 전에 두 장의 부품 수를 사람이 먼저 셈

In [ ]:
%cd /content/OmniParser
from google.colab import files
import fitz
up = files.upload()
os.rename(list(up.keys())[0], 'manual.pdf')

os.makedirs('ikea', exist_ok=True)
doc = fitz.open('manual.pdf')
for i, page in enumerate(doc):
    page.get_pixmap(dpi=150).save(f'ikea/p{i+1:02d}.png')
print(len(doc), '쪽')

## 7. 실험 1: 기본 설정

- Q1 설명서 부품에 박스가 쳐지는가
- Q2 붙은 설명이 쓸모 있는가
- 쪽 번호는 고른 것으로 바꿈

In [ ]:
A = 'ikea/p03.png'   # 부품 목록 페이지
B = 'ikea/p07.png'   # 조립 단계 페이지

la, dfa = run(A, tag='base'); display(la)
lb, dfb = run(B, tag='base'); display(lb)

## 8. 실험 2: 크게 보게 하기

- Q4 크게 보게 하면 작은 부품이 잡히는가
- sz1280_noscale 확인: scale_img=False면 imgsz가 모델에 전달되지 않는 구조로 읽힘 -> 요소 수가 A base와 같으면 imgsz가 무시된다는 뜻
- 메모리 에러가 나면 1280을 960으로

In [ ]:
l1, df1 = run(A, tag='sz1280', scale_img=True, imgsz=1280); display(l1)
l2, df2 = run(A, tag='sz1280_noscale', imgsz=1280)

## 9. 실험 3: 같은 나사 비교

- Q3 같은 모양 나사끼리 구별되는가
- 설명 모델은 잘라낸 그림을 64x64로 줄여서 받음 -> 줄어든 그림에서 사람 눈으로도 구별되는지 확인
- sz1280 그림에서 비슷한 나사 3~4개의 번호를 읽어 아래 리스트에 넣기

In [ ]:
show_crops(A, df1, [5, 6, 7, 8])

## 10. 결과 저장

In [ ]:
!zip -r out.zip out
files.download('out.zip')